# 03 — Proposal, chốt manifest, simulated annotation
**Input:** output NB1. **GPU:** nên có. **Output:** `art/retrieval/`, `art/selections/`.

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
SCENARIO = "missing"   # missing: base chưa có novel class | weak: base có ít ảnh novel
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"   # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
subprocess.run(["git", "-C", R, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
sys.path.insert(0, f"{R}/src")
import numpy as np, pandas as pd
from topicx import common as C, pipeline as P
cfg = C.load_config(SMOKE, SCENARIO)
print(SCENARIO, cfg["art"], "| copy từ input:", C.sync_inputs(cfg), "file")

In [ ]:
STAGE = "retrieval"
# Test protocol (CPU, vài giây). Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE);

## Health check dataset
Vài giây, không đọc cả file nhãn: đường dẫn, số ảnh, định dạng nhãn, ảnh mẫu có vẽ box. Có mục FAIL thì notebook dừng ở đây.

In [ ]:
hc, sample = P.health_check(cfg)
if sample is not None:
    display(sample.resize((640, 360)))   # box đỏ phải ôm đúng vật thể

In [ ]:
B = cfg["branches"] + cfg["analysis_branches"]
pool = P.Pool(cfg)
P.propose(cfg, {b: P.PROPOSERS[b] for b in B if b in P.PROPOSERS}, cfg["seeds"], cfg["ks"], pool)
P.propose_privileged(cfg, B, cfg["seeds"], cfg["ks"], pool)
ret = P.retrieval_eval(cfg, pool)
display(ret.groupby(["k", "branch"])[["precision_at_k", "recall_at_k", "novel_instances", "base_instances", "nn_cos_mean"]].mean())